# CBLV Node Classification with GNN

Build DGL graphs from phylogenetic trees for node-level prediction tasks.

**Features:**
- **Node features**: CBLV encoding (tree topology) per location
- **Edge features**: DTW distance and lag statistics between location pairs
- **Node labels**: R0 and Source_Sink_Score

In [25]:
# Cell 1: Load and Summarize Tree Files

import os, sys, re
import numpy as np
from pathlib import Path
from collections import defaultdict
import dendropy as dp

# Add utils to path
sys.path.insert(0, str(Path(os.getcwd()).parent / 'utils'))

# Configuration
input_folder = Path('/Users/lukelyu/Desktop/epidata/test')
file_pattern = '*_beast2.trees'

# Load and analyze trees
tree_files = sorted(input_folder.glob(file_pattern))
all_trees_info = []
max_tips_per_location = 0
max_tips_info = None

for tree_file in tree_files:
    tree_list = dp.TreeList.get(
        path=str(tree_file), schema='nexus',
        suppress_internal_node_taxa=True, suppress_leaf_node_taxa=True
    )
    
    for idx, phy in enumerate(tree_list):
        phy.is_rooted = True
        phy.suppress_unifurcations()
        phy.calc_node_root_distances()
        
        # Count tips per location
        pop_counts = defaultdict(int)
        for nd in phy.leaf_node_iter():
            if nd.annotations:
                type_annot = nd.annotations.get_value('type')
                if nd.annotations.get_value('samp') == 'sample' and type_annot and '{' in str(type_annot):
                    loc = int(str(type_annot).split('{')[1].split('}')[0])
                    pop_counts[loc] += 1
        
        # Track max tips
        for loc, count in pop_counts.items():
            if count > max_tips_per_location:
                max_tips_per_location = count
                max_tips_info = {'file': tree_file.name, 'tree_idx': idx, 'location': loc}
        
        all_trees_info.append({
            'file': tree_file.name, 'tree_idx': idx,
            'n_tips': len(phy.leaf_nodes()),
            'height': max(nd.root_distance for nd in phy.leaf_node_iter()),
            'pop_counts': dict(pop_counts)
        })

# Summary
print(f"{'='*60}")
print(f"TREE SUMMARY")
print(f"{'='*60}")
print(f"Input folder: {input_folder}")
print(f"Tree files: {len(tree_files)}")
print(f"Total trees: {len(all_trees_info)}")
print(f"Locations per tree: {len(all_trees_info[0]['pop_counts'])}")
print(f"Tips range: {min(t['n_tips'] for t in all_trees_info)} - {max(t['n_tips'] for t in all_trees_info)}")
print(f"Height range: {min(t['height'] for t in all_trees_info):.1f} - {max(t['height'] for t in all_trees_info):.1f}")
print(f"\nMax tips in single population: {max_tips_per_location}")
print(f"  Source: {max_tips_info['file']}, Tree {max_tips_info['tree_idx']}, Location I{{{max_tips_info['location']}}}")
print(f"  → tree_width = {max_tips_per_location}")

TREE SUMMARY
Input folder: /Users/lukelyu/Desktop/epidata/test
Tree files: 5
Total trees: 25
Locations per tree: 16
Tips range: 1700 - 4451
Height range: 251.7 - 760.1

Max tips in single population: 414
  Source: 4_beast2.trees, Tree 4, Location I{6}
  → tree_width = 414


In [26]:
# Cell 2: CBLV Node Feature Encoder

def get_location(node):
    """Extract location from 'type' annotation (e.g., 'I{7}' -> 7)."""
    type_annot = node.annotations.get_value('type') if node.annotations else None
    if type_annot and '{' in str(type_annot):
        return int(str(type_annot).split('{')[1].split('}')[0])
    return None

def is_sample(node):
    """Check if node is an actual sample (not ancestral reconstruction)."""
    return node.annotations and node.annotations.get_value('samp') == 'sample'


class VirtualSubtreeEncoder:
    """
    CBLV encoder using virtual subtree traversal (no tree copying).
    Encodes tree topology as a fixed-size matrix per location.
    """
    def __init__(self, phy, tree_height):
        self.phy = phy
        self.tree_height = tree_height
        self._preprocess()

    def _preprocess(self):
        """Compute location counts and max distances for each node (bottom-up)."""
        for nd in self.phy.postorder_node_iter():
            if nd.is_leaf():
                loc = get_location(nd)
                if is_sample(nd) and loc is not None:
                    nd.loc_counts = {loc: 1}
                    nd.loc_max_dist = {loc: nd.root_distance}
                else:
                    nd.loc_counts, nd.loc_max_dist = {}, {}
            else:
                nd.loc_counts, nd.loc_max_dist = {}, {}
                for child in nd.child_nodes():
                    for loc, count in getattr(child, 'loc_counts', {}).items():
                        nd.loc_counts[loc] = nd.loc_counts.get(loc, 0) + count
                    for loc, dist in getattr(child, 'loc_max_dist', {}).items():
                        nd.loc_max_dist[loc] = max(nd.loc_max_dist.get(loc, 0), dist)

    def _find_mrca(self, target_loc):
        """Find MRCA of all tips with target location."""
        total = self.phy.seed_node.loc_counts.get(target_loc, 0)
        if total < 2:
            return None
        mrca = self.phy.seed_node
        while True:
            children_with_all = [c for c in mrca.child_nodes() if c.loc_counts.get(target_loc, 0) == total]
            if len(children_with_all) == 1:
                mrca = children_with_all[0]
            else:
                break
        return mrca

    def _is_branching_point(self, node, target_loc):
        if node.is_leaf():
            return False
        return sum(1 for c in node.child_nodes() if c.loc_counts.get(target_loc, 0) > 0) >= 2

    def _find_parent_branching_point(self, node, target_loc, mrca):
        current = node.parent_node
        while current:
            if current == mrca or self._is_branching_point(current, target_loc):
                return current
            current = current.parent_node
        return mrca

    def _compute_accumulated_edge(self, node, parent_branch):
        total = 0
        current = node
        while current != parent_branch and current:
            total += current.edge.length or 0
            current = current.parent_node
        return total

    def _virtual_inorder(self, node, target_loc, last_branch_dist, mrca):
        """Generator for virtual in-order traversal of location subtree."""
        if node.is_leaf():
            if is_sample(node) and get_location(node) == target_loc:
                parent_branch = self._find_parent_branching_point(node, target_loc, mrca)
                yield ('leaf', node.root_distance - last_branch_dist, self._compute_accumulated_edge(node, parent_branch))
        elif self._is_branching_point(node, target_loc):
            children = sorted([c for c in node.child_nodes() if c.loc_counts.get(target_loc, 0) > 0],
                            key=lambda c: c.loc_max_dist.get(target_loc, 0), reverse=True)
            yield from self._virtual_inorder(children[0], target_loc, last_branch_dist, mrca)
            accum = node.edge.length or 0 if node == mrca else self._compute_accumulated_edge(node, self._find_parent_branching_point(node, target_loc, mrca))
            yield ('internal', node.root_distance, accum)
            for child in children[1:]:
                yield from self._virtual_inorder(child, target_loc, node.root_distance, mrca)
        else:
            relevant = [c for c in node.child_nodes() if c.loc_counts.get(target_loc, 0) > 0]
            if relevant:
                yield from self._virtual_inorder(relevant[0], target_loc, last_branch_dist, mrca)

    def encode_cblv(self, target_loc, tree_width=None, rescale=True):
        """
        Encode CBLV for a target location.
        Returns: (heights matrix, stem_distance, n_tips)
        """
        mrca = self._find_mrca(target_loc)
        if mrca is None:
            return np.zeros((tree_width or 1, 4)), 0, self.phy.seed_node.loc_counts.get(target_loc, 0)

        stem = mrca.root_distance
        n_tips = mrca.loc_counts.get(target_loc, 0)
        heights = np.zeros((n_tips, 4))
        idx = 0

        for event in self._virtual_inorder(mrca, target_loc, mrca.root_distance, mrca):
            if idx >= n_tips:
                break
            if event[0] == 'leaf':
                heights[idx, 0] = event[1] + (stem if idx == 0 else 0)
                heights[idx, 2] = event[2]
            else:
                if idx + 1 < n_tips:
                    heights[idx + 1, 1] = event[1]
                    heights[idx + 1, 3] = event[2]
                idx += 1

        if rescale:
            heights /= self.tree_height

        # Pad or truncate
        if tree_width and n_tips != tree_width:
            padded = np.zeros((tree_width, 4))
            padded[:min(n_tips, tree_width)] = heights[:min(n_tips, tree_width)]
            heights = padded

        return heights, stem, n_tips

    def get_all_locations(self):
        return sorted(set(get_location(nd) for nd in self.phy.leaf_node_iter() 
                         if get_location(nd) is not None and is_sample(nd)))

print("CBLV encoder ready")

CBLV encoder ready


In [27]:
# Cell 3: DTW Edge Feature Extraction

from scipy.stats import gaussian_kde

def parse_tree_string(tree_file, tree_idx=0):
    """Extract tree string from BEAST2 .trees file."""
    with open(tree_file) as f:
        trees = re.findall(r'tree STATE_\d+ = (.+?)(?=\ntree |\nEnd;|$)', f.read(), re.DOTALL)
    return trees[tree_idx].strip() if trees else None

def extract_tip_times(tree_string):
    """Extract tip times grouped by location."""
    tips = defaultdict(list)
    for loc, time in re.findall(r'\d+\[&type="I\{(\d+)\}",samp="sample",time=([\d.]+)\]', tree_string):
        tips[int(loc)].append(float(time))
    return dict(tips)

def tips_to_curves(tips_by_loc, num_points=200):
    """Convert tip times to KDE-smoothed epidemic curves."""
    all_times = [t for times in tips_by_loc.values() for t in times]
    if not all_times:
        return {}, 0
    
    t_min, t_max = min(all_times), max(all_times)
    t_grid = np.linspace(t_min, t_max, num_points)
    dt = (t_max - t_min) / (num_points - 1)
    
    curves = {}
    for loc, times in tips_by_loc.items():
        curves[loc] = gaussian_kde(times)(t_grid) * len(times) if len(times) >= 2 else np.zeros(num_points)
    return curves, dt

def dtw_distance(curve_a, curve_b):
    """Compute DTW distance and lag statistics between two curves."""
    n, m = len(curve_a), len(curve_b)
    cost = (curve_a[:, None] - curve_b[None, :]) ** 2
    
    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dtw[i, j] = cost[i-1, j-1] + min(dtw[i-1, j-1], dtw[i-1, j], dtw[i, j-1])
    
    # Backtrack for lags
    lags, i, j = [], n, m
    while i > 0 and j > 0:
        lags.append(j - i)
        step = np.argmin([dtw[i-1, j-1], dtw[i-1, j], dtw[i, j-1]])
        if step == 0: i, j = i-1, j-1
        elif step == 1: i -= 1
        else: j -= 1
    
    lags = np.array(lags)
    return dtw[n, m], lags.mean(), lags.std()

def compute_dtw_edge_features(tree_file, tree_idx=0):
    """Compute DTW features for all location pairs."""
    tree_string = parse_tree_string(tree_file, tree_idx)
    if not tree_string:
        return None, None, None, []
    
    tips_by_loc = extract_tip_times(tree_string)
    if not tips_by_loc:
        return None, None, None, []
    
    curves, dt = tips_to_curves(tips_by_loc)
    if not curves:
        return None, None, None, []
    
    locations = sorted(curves.keys())
    curve_arr = np.array([curves[loc] for loc in locations])
    
    src_list, dst_list, edge_feats = [], [], []
    for i in range(len(locations)):
        for j in range(len(locations)):
            if i != j:
                dist, lag, lag_std = dtw_distance(curve_arr[i], curve_arr[j])
                src_list.append(i)
                dst_list.append(j)
                edge_feats.append([dist, lag * dt, lag_std * dt])
    
    return src_list, dst_list, np.array(edge_feats), locations

print("DTW edge features ready")

DTW edge features ready


In [28]:
# Cell 4: DGL Graph Builder with Labels

import dgl
import torch
from tqdm import tqdm
from trajectory_utils import load_trajectory_wide, load_reactions_from_xml, load_R0_and_population_from_csv
from node_feature import classify_events_for_sample, calculate_node_event_metrics


def extract_labels(input_folder, file_prefix, tree_idx, num_nodes):
    """Extract R0 and Source_Sink_Score labels for a specific tree."""
    input_folder = Path(input_folder)
    
    # R0 from parameter CSV
    params = load_R0_and_population_from_csv(str(input_folder / f"{file_prefix}_parameter.csv"))
    r0_values = [params['R0'].get(i, 0.0) for i in range(num_nodes)]
    
    # Source_Sink_Score from trajectory
    df_wide = load_trajectory_wide(str(input_folder / f"{file_prefix}_beast2.traj"))
    reaction_lookup = load_reactions_from_xml(str(input_folder / f"{file_prefix}_beast2.xml"))
    species_cols = [c for c in df_wide.columns if c not in ['Sample', 't']]
    
    sample_data = df_wide[df_wide['Sample'] == tree_idx].sort_values('t').reset_index(drop=True)
    events_df = classify_events_for_sample(sample_data, reaction_lookup, species_cols)
    event_counts = dict(events_df['event_type'].value_counts()) if not events_df.empty else {}
    node_metrics = calculate_node_event_metrics(event_counts, num_nodes)
    
    return {
        'R0': np.array(r0_values, dtype=np.float32),
        'Source_Sink_Score': np.array([node_metrics[i]['source_sink_score'] for i in range(num_nodes)], dtype=np.float32)
    }


def build_dgl_graph(tree_file, tree_idx, tree_width, input_folder):
    """
    Build a DGL graph with CBLV node features, DTW edge features, and labels.
    """
    # Load and preprocess tree
    tree_list = dp.TreeList.get(path=tree_file, schema='nexus',
                                 suppress_internal_node_taxa=True, suppress_leaf_node_taxa=True)
    phy = tree_list[tree_idx]
    phy.is_rooted = True
    phy.suppress_unifurcations()
    phy.calc_node_root_distances()
    tree_height = max(nd.root_distance for nd in phy.leaf_node_iter())
    
    # CBLV node features
    encoder = VirtualSubtreeEncoder(phy, tree_height)
    locations = encoder.get_all_locations()
    n_nodes = len(locations)
    
    node_feats = np.zeros((n_nodes, tree_width * 4))
    for i, loc in enumerate(locations):
        cblv, _, _ = encoder.encode_cblv(loc, tree_width=tree_width, rescale=True)
        node_feats[i] = cblv.flatten()
    
    # DTW edge features
    src_list, dst_list, edge_feats, dtw_locs = compute_dtw_edge_features(tree_file, tree_idx)
    assert locations == dtw_locs, f"Location mismatch: CBLV={locations}, DTW={dtw_locs}"
    
    # Labels
    file_prefix = Path(tree_file).stem.replace('_beast2', '')
    labels = extract_labels(input_folder, file_prefix, tree_idx, n_nodes)
    
    # Build graph
    g = dgl.graph((src_list, dst_list), num_nodes=n_nodes)
    g.ndata['feat'] = torch.tensor(node_feats, dtype=torch.float32)
    g.ndata['location'] = torch.tensor(locations, dtype=torch.long)
    g.ndata['R0'] = torch.tensor(labels['R0'], dtype=torch.float32)
    g.ndata['Source_Sink_Score'] = torch.tensor(labels['Source_Sink_Score'], dtype=torch.float32)
    g.edata['feat'] = torch.tensor(edge_feats, dtype=torch.float32)
    
    return g, locations, tree_height


def build_graphs_from_folder(input_folder, tree_width, file_pattern='*_beast2.trees'):
    """Build DGL graphs from all tree files in a folder."""
    input_folder = Path(input_folder)
    tree_files = sorted(input_folder.glob(file_pattern))
    
    graphs = []
    for tree_file in tqdm(tree_files, desc="Building graphs"):
        file_prefix = tree_file.stem.replace('_beast2', '')
        with open(tree_file) as f:
            n_trees = len(re.findall(r'tree STATE_\d+', f.read()))
        
        for tree_idx in range(n_trees):
            try:
                g, locs, height = build_dgl_graph(str(tree_file), tree_idx, tree_width, input_folder)
                graphs.append((g, f"{file_prefix}_{tree_idx}", locs, height))
            except Exception as e:
                print(f"Error {tree_file.name} tree {tree_idx}: {e}")
    
    return graphs


def normalize_edge_features(graphs):
    """Z-score normalize edge features across all graphs."""
    all_feats = torch.cat([g.edata['feat'] for g, _, _, _ in graphs], dim=0)
    
    for i in range(all_feats.shape[1]):
        mean, std = all_feats[:, i].mean(), all_feats[:, i].std()
        for g, _, _, _ in graphs:
            g.edata['feat'][:, i] = (g.edata['feat'][:, i] - mean) / (std + 1e-8)
    
    return graphs

print("Graph builder ready")

Graph builder ready


In [29]:
# Cell 5: Build Graphs

tree_width = max_tips_per_location
graphs = build_graphs_from_folder(input_folder, tree_width=tree_width)
graphs = normalize_edge_features(graphs)

# Summary
g0, _, _, _ = graphs[0]
print(f"\n{'='*60}")
print(f"GRAPH SUMMARY")
print(f"{'='*60}")
print(f"Total graphs: {len(graphs)}")
print(f"Nodes per graph: {g0.num_nodes()}")
print(f"Edges per graph: {g0.num_edges()}")
print(f"\nNode features (CBLV):")
print(f"  Shape: {g0.ndata['feat'].shape} ({tree_width} × 4 flattened)")
print(f"  Range: [{g0.ndata['feat'].min():.4f}, {g0.ndata['feat'].max():.4f}]")
print(f"  Non-zero: {(g0.ndata['feat'] != 0).float().mean():.1%}")
print(f"\nEdge features (DTW, Z-score normalized):")
print(f"  Shape: {g0.edata['feat'].shape}")
all_edge = torch.cat([g.edata['feat'] for g, _, _, _ in graphs])
print(f"  dtw_distance:  [{all_edge[:,0].min():.2f}, {all_edge[:,0].max():.2f}]")
print(f"  dtw_lag_mean:  [{all_edge[:,1].min():.2f}, {all_edge[:,1].max():.2f}]")
print(f"  dtw_lag_std:   [{all_edge[:,2].min():.2f}, {all_edge[:,2].max():.2f}]")
print(f"\nNode labels:")
all_r0 = torch.cat([g.ndata['R0'] for g, _, _, _ in graphs])
all_sss = torch.cat([g.ndata['Source_Sink_Score'] for g, _, _, _ in graphs])
print(f"  R0: [{all_r0.min():.4f}, {all_r0.max():.4f}]")
print(f"  Source_Sink_Score: [{all_sss[~all_sss.isnan()].min():.4f}, {all_sss[~all_sss.isnan()].max():.4f}]")

Building graphs: 100%|████████████████████████████| 5/5 [05:34<00:00, 66.82s/it]


GRAPH SUMMARY
Total graphs: 25
Nodes per graph: 16
Edges per graph: 240

Node features (CBLV):
  Shape: torch.Size([16, 1656]) (414 × 4 flattened)
  Range: [0.0000, 1.0000]
  Non-zero: 31.4%

Edge features (DTW, Z-score normalized):
  Shape: torch.Size([240, 3])
  dtw_distance:  [-0.67, 5.58]
  dtw_lag_mean:  [-4.69, 4.69]
  dtw_lag_std:   [-1.70, 5.47]

Node labels:
  R0: [2.4959, 5.3393]
  Source_Sink_Score: [-0.3914, 0.3643]


In [30]:
# Cell 6: Inspect Sample Graph

g, graph_id, locs, height = graphs[0]

print(f"{'='*60}")
print(f"SAMPLE GRAPH: {graph_id}")
print(f"{'='*60}")
print(f"Tree height: {height:.2f}")
print(f"Locations: {locs}")

print(f"\n{'Node':<6} {'R0':<8} {'SSS':<10} {'CBLV[0:4]'}")
print("-" * 50)
for i in range(g.num_nodes()):
    r0 = g.ndata['R0'][i].item()
    sss = g.ndata['Source_Sink_Score'][i].item()
    cblv = g.ndata['feat'][i, :4].tolist()
    sss_str = f"{sss:.4f}" if not np.isnan(sss) else "NaN"
    print(f"{locs[i]:<6} {r0:<8.4f} {sss_str:<10} [{cblv[0]:.3f}, {cblv[1]:.3f}, {cblv[2]:.3f}, {cblv[3]:.3f}]")

SAMPLE GRAPH: 0_0
Tree height: 270.27
Locations: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

Node   R0       SSS        CBLV[0:4]
--------------------------------------------------
0      4.1860   0.1082     [0.912, 0.000, 0.804, 0.000]
1      5.1873   -0.1797    [0.750, 0.000, 0.584, 0.000]
2      4.0059   0.1077     [0.779, 0.000, 0.688, 0.000]
3      4.7149   0.0679     [0.848, 0.000, 0.687, 0.000]
4      4.4945   -0.3199    [0.784, 0.000, 0.684, 0.000]
5      3.8058   -0.0918    [0.863, 0.000, 0.658, 0.000]
6      5.2039   0.1336     [0.701, 0.000, 0.475, 0.000]
7      3.6501   -0.0089    [0.760, 0.000, 0.523, 0.000]
8      4.6974   -0.1829    [0.755, 0.000, 0.651, 0.000]
9      5.3393   0.1660     [0.836, 0.000, 0.689, 0.000]
10     3.7237   0.0537     [0.831, 0.000, 0.305, 0.000]
11     4.3593   -0.3012    [0.792, 0.000, 0.635, 0.000]
12     5.0149   0.0121     [0.859, 0.000, 0.674, 0.000]
13     4.2302   0.1903     [0.637, 0.000, 0.434, 0.000]
14     5.3172   0.0088 

In [31]:
# Cell 7: Validation

def validate_graph(g, graph_id, input_folder, tree_width):
    """Validate graph features and labels against source files."""
    file_prefix, tree_idx = graph_id.rsplit('_', 1)
    tree_idx = int(tree_idx)
    num_nodes = g.num_nodes()
    errors = []
    
    # 1. Validate R0
    params = load_R0_and_population_from_csv(str(input_folder / f"{file_prefix}_parameter.csv"))
    r0_valid = all(abs(params['R0'].get(i, 0) - g.ndata['R0'][i].item()) < 1e-4 for i in range(num_nodes))
    
    # 2. Validate Source_Sink_Score
    df_wide = load_trajectory_wide(str(input_folder / f"{file_prefix}_beast2.traj"))
    reaction_lookup = load_reactions_from_xml(str(input_folder / f"{file_prefix}_beast2.xml"))
    species_cols = [c for c in df_wide.columns if c not in ['Sample', 't']]
    sample_data = df_wide[df_wide['Sample'] == tree_idx].sort_values('t').reset_index(drop=True)
    events_df = classify_events_for_sample(sample_data, reaction_lookup, species_cols)
    event_counts = dict(events_df['event_type'].value_counts()) if not events_df.empty else {}
    node_metrics = calculate_node_event_metrics(event_counts, num_nodes)
    
    sss_valid = True
    for i in range(num_nodes):
        expected, actual = node_metrics[i]['source_sink_score'], g.ndata['Source_Sink_Score'][i].item()
        if not (np.isnan(expected) and np.isnan(actual)) and abs(expected - actual) > 1e-4:
            sss_valid = False
    
    # 3. Validate CBLV
    tree_list = dp.TreeList.get(path=str(input_folder / f"{file_prefix}_beast2.trees"), schema='nexus',
                                 suppress_internal_node_taxa=True, suppress_leaf_node_taxa=True)
    phy = tree_list[tree_idx]
    phy.is_rooted = True
    phy.suppress_unifurcations()
    phy.calc_node_root_distances()
    tree_height = max(nd.root_distance for nd in phy.leaf_node_iter())
    encoder = VirtualSubtreeEncoder(phy, tree_height)
    locations = encoder.get_all_locations()
    
    cblv_valid = True
    for i, loc in enumerate(locations):
        expected, _, _ = encoder.encode_cblv(loc, tree_width=tree_width, rescale=True)
        actual = g.ndata['feat'][i].numpy().reshape(tree_width, 4)
        if not np.allclose(expected, actual, rtol=1e-4):
            cblv_valid = False
    
    cblv_range = (g.ndata['feat'].min().item(), g.ndata['feat'].max().item())
    
    # 4. Validate edges
    edge_valid = g.num_edges() == num_nodes * (num_nodes - 1) and g.edata['feat'].shape[1] == 3
    
    return {'r0': r0_valid, 'sss': sss_valid, 'cblv': cblv_valid, 'edge': edge_valid, 'cblv_range': cblv_range}


# Run validation
print(f"{'='*60}")
print(f"VALIDATION")
print(f"{'='*60}")

n_validate = min(5, len(graphs))
results = []

for i in range(n_validate):
    g, graph_id, _, _ = graphs[i]
    r = validate_graph(g, graph_id, input_folder, tree_width)
    results.append(r)
    status = '✓' if all([r['r0'], r['sss'], r['cblv'], r['edge']]) else '✗'
    print(f"{graph_id}: R0={'✓' if r['r0'] else '✗'} SSS={'✓' if r['sss'] else '✗'} "
          f"CBLV={'✓' if r['cblv'] else '✗'}[{r['cblv_range'][0]:.2f},{r['cblv_range'][1]:.2f}] "
          f"Edge={'✓' if r['edge'] else '✗'} → {status}")

print(f"\n{'='*60}")
print(f"VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"R0 labels:           {sum(r['r0'] for r in results)}/{n_validate}")
print(f"Source_Sink_Score:   {sum(r['sss'] for r in results)}/{n_validate}")
print(f"CBLV features:       {sum(r['cblv'] for r in results)}/{n_validate}")
print(f"Edge structure:      {sum(r['edge'] for r in results)}/{n_validate}")
all_pass = all(r['r0'] and r['sss'] and r['cblv'] and r['edge'] for r in results)
print(f"\nResult: {'ALL PASS ✓' if all_pass else 'SOME FAILURES ✗'}")

VALIDATION
0_0: R0=✓ SSS=✓ CBLV=✓[0.00,1.00] Edge=✓ → ✓
0_1: R0=✓ SSS=✓ CBLV=✓[0.00,1.00] Edge=✓ → ✓
0_2: R0=✓ SSS=✓ CBLV=✓[0.00,1.00] Edge=✓ → ✓
0_3: R0=✓ SSS=✓ CBLV=✓[0.00,1.00] Edge=✓ → ✓
0_4: R0=✓ SSS=✓ CBLV=✓[0.00,1.00] Edge=✓ → ✓

VALIDATION SUMMARY
R0 labels:           5/5
Source_Sink_Score:   5/5
CBLV features:       5/5
Edge structure:      5/5

Result: ALL PASS ✓
